# [연습] 문제 [1] 포켓몬 능력치(`Total`) 예측 (회귀)

- 이름 : 이우진
- 제출용이 아닌 **연습 파일**입니다. (제출은 문제 [2]로 진행)
- 능력치 이외의 정보(타입, 세대, 키, 몸무게 등)로 `Total`을 예측하는 회귀 모델을 만듭니다.

---
## 해결 과제 (공통)

### 1. `pandas`를 사용하여 `pokemon_alopez247.csv` 파일 데이터를 불러와 `DataFrame`으로 저장하시오.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("pokemon_alopez247.csv")

pd.set_option("display.max_columns", None)

print(df.shape)

### 2. 저장된 데이터에서 상위 5개 행을 출력하시오.

In [ ]:
df.head()


### 3. 각 열의 이름, 결측치 여부, 데이터 타입(`dtype`)을 한 번에 확인하시오.

`df.info()`를 실행한 후, 출력 결과를 바탕으로 다음을 기술하시오.
- 결측치가 존재하는 열과 결측 개수
- 각 열의 결측이 **무엇을 의미하는지** (위 데이터 설명 참고)

In [ ]:
df.info()

**결측치가 존재하는 열과 결측 개수는 다음과 같습니다.**

| 열 | Non-Null | 결측 개수 |
|---|---|---|
| `Type_2` | 350 | 371 |
| `Pr_Male` | 644 | 77 |
| `Egg_Group_2` | 191 | 530 |

**각 열의 결측이 의미하는 것은 다음과 같습니다.**

- `Type_2` : 보조 타입이 **없는** 포켓몬입니다. (타입이 하나뿐입니다.)
- `Pr_Male` : 성별 구분이 **없는** 포켓몬입니다. (`hasGender`가 False인 77마리와 정확히 일치합니다.)
- `Egg_Group_2` : 두 번째 알 그룹이 **없는** 포켓몬입니다.

→ 세 열 모두 값을 몰라서 비어 있는 것이 아니라, **"해당 사항 없음"을 뜻하는 결측**입니다.

### 4. 결측치를 처리하시오.

- 처리 방식은 자유롭게 선택해도 됩니다. (예: `fillna()`로 `'None'` 또는 특정 숫자 채우기)
- **왜 그렇게 처리했는지 한 줄로 기술**하시오.
- 처리 후 전체 결측치 개수가 0인지 확인하시오.

In [ ]:
# 범주형 : 보조 타입, 두 번째 알 그룹이 없는 경우이므로 "None"이라는 하나의 범주로 채웁니다.
df["Type_2"] = df["Type_2"].fillna("None")
df["Egg_Group_2"] = df["Egg_Group_2"].fillna("None")

# 수치형 : 성별이 없는 포켓몬이므로 확률(0~1)에 없는 값인 -1로 채워 구분합니다.
df["Pr_Male"] = df["Pr_Male"].fillna(-1)

print("전체 결측치 개수 :", df.isna().sum().sum())

**결측치 처리 방법과 이유 (한 줄 요약)** : 세 열의 결측은 값을 모르는 것이 아니라 "해당 없음"을 뜻하기 때문에, 범주형(`Type_2`, `Egg_Group_2`)은 `"None"` 범주로, 수치형(`Pr_Male`)은 확률 범위(0~1)에 없는 `-1`로 채워 "없음" 자체를 정보로 남겼습니다.

**자세한 설명**

1. **결측의 의미** : 이 데이터의 결측은 값을 기록하지 못해서 비어 있는 것이 아니라, 원래 값이 없어서 비어 있는 것입니다. 예를 들어 파이리는 타입이 불꽃 하나뿐이라 `Type_2`가 비어 있고, 성별이 없는 포켓몬은 "수컷일 확률" 자체가 없기 때문에 `Pr_Male`이 비어 있습니다. 따라서 평균값으로 채우거나 행을 삭제하는 방법은 맞지 않다고 판단했습니다. (행을 삭제하면 `Egg_Group_2` 때문에 721마리 중 530마리가 사라집니다.)

2. **범주형 (`Type_2`, `Egg_Group_2`)** : 빈칸에 `"None"`이라는 값을 넣어 "보조 타입 없음", "두 번째 알 그룹 없음"을 하나의 범주로 만들었습니다. 원-핫 인코딩을 하면 `Type_2_None` 같은 열이 생기기 때문에, 모델이 "보조 타입이 없는 포켓몬"이라는 정보도 학습할 수 있습니다.

3. **수치형 (`Pr_Male`)** : 숫자 컬럼이라 글자를 넣을 수 없기 때문에, 확률(0~1)로는 나올 수 없는 값인 `-1`로 채웠습니다. 0.5는 수컷일 확률 50%, 0은 항상 암컷을 뜻하지만 `-1`은 "성별 자체가 없음"을 뜻하므로 다른 값과 헷갈리지 않습니다. 만약 평균(약 0.55)으로 채웠다면 성별이 없는 포켓몬이 "수컷일 확률 55%"인 포켓몬처럼 바뀌어 버립니다.

4. **"없음"을 정보로 남긴 이유** : 전설 포켓몬은 성별이 없는 경우가 많기 때문에, "성별이 없다"는 사실 자체가 전설 여부를 예측하는 데 도움이 되는 정보입니다. 결측을 지우거나 엉뚱한 값으로 채우면 이 정보가 사라지므로, "없음"이 그대로 드러나도록 처리했습니다.

### 5. 전설 포켓몬(`isLegendary`=`True`)과 일반 포켓몬이 각각 몇 마리인지, 전설 포켓몬의 비율(%)은 얼마인지 계산하시오.

In [ ]:
counts = df["isLegendary"].value_counts()

legend_table = pd.DataFrame({
    "구분": ["일반 포켓몬", "전설 포켓몬"],
    "마리 수": [counts[False], counts[True]],
    "비율(%)": [counts[False] / len(df) * 100, counts[True] / len(df) * 100],
}).round(2)

display(legend_table)

---
## 해결 과제 (선택) - 문제 [1] 포켓몬 능력치(`Total`) 예측 (회귀)

능력치 이외의 정보(타입, 세대, 키, 몸무게 등)로 `Total`을 예측하는 회귀 모델을 만든다.

### [1-1] `Total`을 타깃(`y`)으로 하고, 사용할 feature(`X`)를 정하시오.

- **`Total`과 여섯 능력치(`HP`, `Attack`, `Defense`, `Sp_Atk`, `Sp_Def`, `Speed`)는 feature로 사용하지 않는다.**
- `Total`은 여섯 능력치를 더한 값이다. 능력치를 feature에 넣으면 정답이 입력에 그대로 들어가는 것이므로(데이터 누수) 예측 모델이라고 할 수 없다.
- **사용 가능한 컬럼:** `Type_1`, `Type_2`, `Generation`, `isLegendary`, `Color`, `hasGender`, `Pr_Male`, `Egg_Group_1`, `Egg_Group_2`, `hasMegaEvolution`, `Height_m`, `Weight_kg`, `Catch_Rate`, `Body_Style`

In [ ]:
# 능력치 6개와 Total은 데이터 누수가 되므로 제외합니다.
FEATURES = [
    "Type_1", "Type_2", "Generation", "isLegendary", "Color",
    "hasGender", "Pr_Male", "Egg_Group_1", "Egg_Group_2",
    "hasMegaEvolution", "Height_m", "Weight_kg", "Catch_Rate", "Body_Style",
]

y = df["Total"]
X = df[FEATURES].copy()

print(f"피처 {len(FEATURES)}개 : {FEATURES}")

**사용한 피처는 다음 14개입니다.**

`Type_1`, `Type_2`, `Generation`, `isLegendary`, `Color`, `hasGender`, `Pr_Male`, `Egg_Group_1`, `Egg_Group_2`, `hasMegaEvolution`, `Height_m`, `Weight_kg`, `Catch_Rate`, `Body_Style`

- `Number`, `Name`은 식별용이고, `Total`은 타깃이며, 여섯 능력치(`HP`, `Attack`, `Defense`, `Sp_Atk`, `Sp_Def`, `Speed`)는 더하면 `Total`이 되어 데이터 누수가 생기기 때문에 피처에서 제외했습니다.

### [1-2] 범주형 컬럼을 숫자로 변환하시오.

- `pd.get_dummies()` 등을 사용해도 됩니다.
- 변환 후 `X`의 컬럼 개수를 출력하시오.

In [ ]:
cat_cols = ["Type_1", "Type_2", "Color", "Egg_Group_1", "Egg_Group_2", "Body_Style"]

# 범주형 컬럼을 원-핫 인코딩으로 0/1 열로 펼칩니다.
X = pd.get_dummies(X, columns=cat_cols, dtype=int)

# True/False 열을 1/0으로 변환합니다.
bool_cols = ["isLegendary", "hasGender", "hasMegaEvolution"]
X[bool_cols] = X[bool_cols].astype(int)

print("변환 후 X의 컬럼 개수 :", X.shape[1])

### [1-3] 수치형 변수(`Generation`, `Height_m`, `Weight_kg`, `Catch_Rate`, `Pr_Male`)와 `Total`의 상관계수를 구하시오.

- `df[[...]].corr()['Total']` 형식으로 계산하시오.
- 상관계수 절댓값이 가장 큰 변수 3개를 기술하시오.

In [ ]:
num_cols = ["Generation", "Height_m", "Weight_kg", "Catch_Rate", "Pr_Male"]

corr = df[num_cols + ["Total"]].corr()["Total"].drop("Total")

corr_table = pd.DataFrame({
    "상관계수": corr,
    "절댓값": corr.abs(),
}).sort_values("절댓값", ascending=False).round(2)

display(corr_table)

#### (추가 분석) 상관계수 막대그래프

In [ ]:
# (추가 분석) 수치형 변수와 Total의 상관계수를 막대그래프로 비교합니다.
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"     # 한글 폰트를 설정합니다.
plt.rcParams["axes.unicode_minus"] = False        # 마이너스 기호가 깨지지 않게 합니다.

corr_sorted = corr.reindex(corr.abs().sort_values().index)   # 절댓값이 큰 변수가 위로 오게 정렬합니다.
bar_colors = ["#2a78d6" if v > 0 else "#eb6834" for v in corr_sorted]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(corr_sorted.index, corr_sorted.values, color=bar_colors, edgecolor="white")
ax.bar_label(bars, fmt="%.2f", padding=3)

ax.axvline(0, color="#52514e", lw=1)
ax.set_xlim(-1, 1)
ax.set_xlabel("Total과의 상관계수 (파랑 : 양의 상관, 주황 : 음의 상관)")
ax.set_title("수치형 변수와 Total의 상관계수")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

**상관계수 절댓값이 가장 큰 변수 3개는 다음과 같습니다.**

| 순위 | 변수 | 상관계수 | 해석 |
|---|---|---|---|
| 1 | `Catch_Rate` | -0.74 | 포획률이 낮을수록(잡기 어려울수록) 능력치가 높습니다. |
| 2 | `Weight_kg` | 0.54 | 무거울수록 능력치가 높은 경향이 있습니다. |
| 3 | `Height_m` | 0.53 | 키가 클수록 능력치가 높은 경향이 있습니다. |

- `Catch_Rate`는 음의 상관이지만 절댓값이 가장 커서, `Total`과 가장 강하게 관련된 변수입니다.
- `Pr_Male`(-0.32)은 결측치를 `-1`로 채운 영향이 있습니다. 성별이 없는 포켓몬에 능력치가 높은 전설이 많아서, `-1`인 포켓몬의 `Total`이 높게 나타납니다.
- `Generation`(0.09)은 `Total`과 거의 관계가 없습니다.
- 위 막대그래프를 보면 `Catch_Rate` 막대가 가장 길고 방향이 반대(음수)라서, 다른 변수보다 `Total`과의 관계가 훨씬 강하다는 것을 한눈에 확인할 수 있습니다.

### [1-4] 학습 데이터와 검증 데이터를 분리하고, 각각의 크기(`shape`)를 출력하시오.

- `train_test_split(X, y, test_size=0.2, random_state=42)`

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

display(pd.DataFrame({
    "데이터": ["학습 X", "검증 X", "학습 y", "검증 y"],
    "shape": [X_train.shape, X_test.shape, y_train.shape, y_test.shape],
}))

### [1-5] `LinearRegression` 모델을 학습시키고, 검증 데이터에 대한 **R²**와 **RMSE**를 계산하시오.

- `r2_score`, `mean_squared_error` (RMSE는 제곱근을 취하여 계산)
- 참고: `Total`의 표준편차는 약 110이다. 평균값만 예측하면 RMSE가 약 110, R²가 0 근처이다.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

lin = LinearRegression()
lin.fit(X_train, y_train)
pred_lin = lin.predict(X_test)

r2_lin = r2_score(y_test, pred_lin)
rmse_lin = np.sqrt(mean_squared_error(y_test, pred_lin))   # MSE에 제곱근을 취해 RMSE를 구합니다.

display(pd.DataFrame({"R²": [r2_lin], "RMSE": [rmse_lin]},
                     index=["LinearRegression"]).round(2))

### [1-6] `RandomForestRegressor` 모델을 학습시키고, 검증 데이터에 대한 **R²**와 **RMSE**를 계산하시오.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

r2_rf = r2_score(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))

display(pd.DataFrame({"R²": [r2_rf], "RMSE": [rmse_rf]},
                     index=["RandomForestRegressor"]).round(2))

#### (추가 분석) 실제값 vs 예측값 산점도

In [ ]:
# (추가 분석) 두 모델의 실제값과 예측값을 산점도로 비교합니다.
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

lims = [y_test.min() - 20, y_test.max() + 20]

for ax, (name, pred, r2, rmse, color) in zip(axes, [
    ("LinearRegression",      pred_lin, r2_lin, rmse_lin, "#2a78d6"),
    ("RandomForestRegressor", pred_rf,  r2_rf,  rmse_rf,  "#eb6834"),
]):
    ax.scatter(y_test, pred, s=30, alpha=0.6, color=color, edgecolor="white", linewidth=0.5)
    ax.plot(lims, lims, color="#0b0b0b", ls="--", lw=1.5, label="완벽한 예측 (y = x)")

    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_title(f"{name}\nR² {r2:.2f} / RMSE {rmse:.2f}")
    ax.set_xlabel("실제 Total")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")

axes[0].set_ylabel("예측 Total")
fig.suptitle("실제값 vs 예측값 (점이 대각선에 가까울수록 잘 맞춘 것)", fontsize=13)
plt.tight_layout()
plt.show()

### [1-7] 두 모델의 성능을 표로 정리하고, 성능 목표 달성 여부를 확인하시오.

**성능 목표**

| 모델 | R² | RMSE |
| --- | --- | --- |
| `LinearRegression` | 0.60 이상 | 70 이하 |
| `RandomForestRegressor` | 0.75 이상 | 55 이하 |

- 두 모델 중 어느 쪽이 더 잘 맞았는지, 그 이유는 무엇일지 기술하시오.

In [ ]:
results = pd.DataFrame([
    {"모델": "LinearRegression",      "R²": r2_lin, "RMSE": rmse_lin, "목표 R²": 0.60, "목표 RMSE": 70},
    {"모델": "RandomForestRegressor", "R²": r2_rf,  "RMSE": rmse_rf,  "목표 R²": 0.75, "목표 RMSE": 55},
]).round(2)

# R²는 목표 이상, RMSE는 목표 이하이면 달성입니다.
results["목표 달성"] = np.where(
    (results["R²"] >= results["목표 R²"]) & (results["RMSE"] <= results["목표 RMSE"]), "O", "X"
)

display(results.set_index("모델"))

#### (추가 분석) 두 모델 성능과 목표 비교 그래프

In [ ]:
# (추가 분석) 두 모델의 R², RMSE를 목표선과 함께 막대그래프로 비교합니다.
model_names = results["모델"]
x = np.arange(len(model_names))
colors = ["#2a78d6", "#eb6834"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# R² : 높을수록 좋음 (목표 이상이면 달성)
# 막대 위 숫자가 목표선과 겹치지 않도록 값은 막대 가운데에 표시합니다.
bars = axes[0].bar(x, results["R²"], color=colors, edgecolor="white", width=0.55)
axes[0].bar_label(bars, fmt="%.2f", label_type="center", color="white", fontsize=12, fontweight="bold")
for k, target in enumerate(results["목표 R²"]):
    axes[0].hlines(target, k - 0.35, k + 0.35, color="#0b0b0b", ls="--", lw=1.5,
                   label="모델별 목표" if k == 0 else None)
axes[0].set_ylim(0, 1)
axes[0].set_title("R² (높을수록 좋음)")

# RMSE : 낮을수록 좋음 (목표 이하이면 달성)
bars = axes[1].bar(x, results["RMSE"], color=colors, edgecolor="white", width=0.55)
axes[1].bar_label(bars, fmt="%.2f", label_type="center", color="white", fontsize=12, fontweight="bold")
for k, target in enumerate(results["목표 RMSE"]):
    axes[1].hlines(target, k - 0.35, k + 0.35, color="#0b0b0b", ls="--", lw=1.5,
                   label="모델별 목표" if k == 0 else None)
axes[1].axhline(y.std(), color="#8a8984", ls=":", lw=1.5, label=f"평균만 예측 시 (약 {y.std():.0f})")
axes[1].set_ylim(0, 130)
axes[1].set_title("RMSE (낮을수록 좋음)")

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(model_names)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)

fig.suptitle("두 모델 성능과 목표 비교", fontsize=13)
plt.tight_layout()
plt.show()

**두 모델의 성능과 목표 달성 여부는 다음과 같습니다.**

| 모델 | R² | RMSE | 목표 달성 |
|---|---|---|---|
| `LinearRegression` | 0.63 | 66.95 | O |
| `RandomForestRegressor` | 0.79 | 50.31 | O |

- 두 모델 모두 목표를 달성했고, 평균값만 예측했을 때(RMSE 약 110, R² 0)보다 훨씬 잘 맞았습니다.
- 위 막대그래프를 보면 두 모델 모두 R²는 목표선 위, RMSE는 목표선 아래에 있고, RMSE는 평균만 예측했을 때(점선)보다 크게 낮은 것을 확인할 수 있습니다.
- 위 산점도를 보면 `RandomForestRegressor`의 점들이 대각선(y = x) 근처에 더 촘촘히 모여 있습니다. 반면 `LinearRegression`은 실제 `Total`이 낮은 포켓몬(200~300)은 높게, 높은 포켓몬(500 이상)은 낮게 예측하는 경향이 보입니다.
- **`RandomForestRegressor`가 더 잘 맞았습니다.** `Catch_Rate`, 키, 몸무게와 `Total`의 관계는 직선 하나로 표현되지 않는 부분이 많습니다. 예를 들어 포획률이 3인 포켓몬은 대부분 능력치가 매우 높지만, 그 사이 구간은 들쭉날쭉합니다. 랜덤포레스트는 여러 트리가 구간을 나눠 이런 비선형 관계를 잡을 수 있기 때문에, 직선 관계만 표현하는 선형회귀보다 오차가 작았던 것으로 보입니다.

### [1-8] (선택) 확인 실험

- 여섯 능력치를 feature에 추가하여 `LinearRegression`을 다시 학습시키고, R²가 왜 그렇게 나오는지 기술하시오. (제출 모델과 별도의 셀에서 진행)
- `Catch_Rate`를 feature에서 제외하고 두 모델의 성능이 어떻게 달라지는지 확인하시오.

In [ ]:
# 실험 1 : 여섯 능력치를 피처에 추가 (데이터 누수 확인용)
stats = ["HP", "Attack", "Defense", "Sp_Atk", "Sp_Def", "Speed"]
X_leak = pd.concat([X, df[stats]], axis=1)

Xl_train, Xl_test, yl_train, yl_test = train_test_split(X_leak, y, test_size=0.2, random_state=42)

lin_leak = LinearRegression().fit(Xl_train, yl_train)
pred_leak = lin_leak.predict(Xl_test)

display(pd.DataFrame({
    "R²": [r2_score(yl_test, pred_leak)],
    "RMSE": [np.sqrt(mean_squared_error(yl_test, pred_leak))],
}, index=["LinearRegression + 능력치 6개"]).round(2))

**여섯 능력치를 추가하면 R²가 1.00, RMSE가 0.00이 나옵니다.**

`Total`은 여섯 능력치를 더한 값(`Total = HP + Attack + Defense + Sp_Atk + Sp_Def + Speed`)이기 때문에, 선형회귀는 각 능력치의 계수를 1로 두기만 하면 정답을 그대로 계산할 수 있습니다. 즉 모델이 무언가를 "예측"한 것이 아니라 정답을 입력으로 받아 더한 것뿐이므로, 이것은 **데이터 누수**이고 의미 있는 예측 모델이라고 할 수 없습니다.

In [ ]:
# 실험 2 : Catch_Rate를 피처에서 제외
X_no_cr = X.drop(columns=["Catch_Rate"])

Xn_train, Xn_test, yn_train, yn_test = train_test_split(X_no_cr, y, test_size=0.2, random_state=42)

rows = []
for name, model in [("LinearRegression", LinearRegression()),
                    ("RandomForestRegressor", RandomForestRegressor(random_state=42))]:
    pred = model.fit(Xn_train, yn_train).predict(Xn_test)
    rows.append({"모델": name,
                 "R² (Catch_Rate 포함)": r2_lin if name == "LinearRegression" else r2_rf,
                 "R² (Catch_Rate 제외)": r2_score(yn_test, pred),
                 "RMSE (Catch_Rate 포함)": rmse_lin if name == "LinearRegression" else rmse_rf,
                 "RMSE (Catch_Rate 제외)": np.sqrt(mean_squared_error(yn_test, pred))})

display(pd.DataFrame(rows).set_index("모델").round(2))

#### (추가 분석) Catch_Rate 포함 / 제외 R² 비교 그래프

In [ ]:
# (추가 분석) Catch_Rate 포함 / 제외 시 R²를 막대그래프로 비교합니다.
cr_table = pd.DataFrame(rows).set_index("모델")

x = np.arange(len(cr_table))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 4.5))
bars_in = ax.bar(x - width / 2, cr_table["R² (Catch_Rate 포함)"], width,
                 color="#2a78d6", edgecolor="white", label="Catch_Rate 포함")
bars_out = ax.bar(x + width / 2, cr_table["R² (Catch_Rate 제외)"], width,
                  color="#eb6834", edgecolor="white", label="Catch_Rate 제외")
ax.bar_label(bars_in, fmt="%.2f", padding=3)
ax.bar_label(bars_out, fmt="%.2f", padding=3)

ax.set_xticks(x)
ax.set_xticklabels(cr_table.index)
ax.set_ylim(0, 1)
ax.set_ylabel("R²")
ax.set_title("Catch_Rate를 빼면 R²가 얼마나 떨어질까?")
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

**`Catch_Rate`를 제외하면 두 모델 모두 성능이 떨어집니다.**

| 모델 | R² (포함 → 제외) | RMSE (포함 → 제외) |
|---|---|---|
| `LinearRegression` | 0.63 → 0.35 | 66.95 → 88.64 |
| `RandomForestRegressor` | 0.79 → 0.70 | 50.31 → 59.93 |

- `Catch_Rate`는 `Total`과 상관계수 절댓값이 가장 큰 변수(-0.74)라서, 빼면 성능이 크게 떨어집니다.
- 특히 `LinearRegression`은 R²가 0.63에서 0.35로 거의 절반이 되었습니다. 선형회귀는 `Catch_Rate`의 직선 관계에 크게 의존하고 있었기 때문입니다.
- 위 그래프를 보면 파란 막대(포함)와 주황 막대(제외)의 차이가 `LinearRegression`에서 훨씬 큰 것을 확인할 수 있습니다.
- `RandomForestRegressor`는 0.79에서 0.70으로 덜 떨어졌습니다. 키, 몸무게, 전설 여부 같은 다른 피처의 조합으로 어느 정도 보완할 수 있었던 것으로 보입니다.
- 두 모델 모두 `Catch_Rate` 없이는 성능 목표를 달성하지 못합니다.

---
## 알게 된 점 / 어려웠던 점

**알게 된 점**
- 능력치 6개를 피처에 넣으면 R²가 1.00이 나와서, 정답을 그대로 계산할 수 있는 정보가 입력에 들어가면 성능이 좋아 보여도 의미 없는 모델이 된다는 것(데이터 누수)을 직접 확인할 수 있었습니다.
- `Catch_Rate` 하나를 빼자 선형회귀의 R²가 0.63에서 0.35로 크게 떨어져서, 상관계수가 큰 변수가 모델 성능에 얼마나 큰 영향을 주는지 알게 되었습니다.

**어려웠던 점**
- `LinearRegression`의 성능이 목표(R² 0.60, RMSE 70)에 아슬아슬하게 걸쳐 있어서, 결측치를 어떻게 채우고 어떤 피처를 넣느냐에 따라 목표 달성 여부가 바뀔 수 있다는 점이 어려웠습니다.